**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Audio & Speech DSP

The most tangible application of everything in the DSP track: sound. We synthesize, analyze, and mangle audio with the tools you already own — spectrograms, filters, and source-filter models. Every cell produces a signal you can export and *listen to* (`scipy.io.wavfile.write`; in Jupyter, `IPython.display.Audio(x, rate=fs)`).

## 1. Pre-requisites

- [Foundations of Signal Processing 1](./Foundations_of_Signal_Processing_1.ipynb) S6 (STFT).
- [Filter Design](./Filter_Design.ipynb).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)
fs = 16_000

def show_spec(x, title="", fs=fs):
    f, t, S = sig.stft(x, fs=fs, nperseg=512)
    plt.figure(figsize=(8.5, 2.8))
    plt.pcolormesh(t, f, 20*np.log10(np.abs(S) + 1e-8), shading="auto", vmin=-100, vmax=-20)
    plt.ylabel("Hz"); plt.xlabel("s"); plt.title(title); plt.colorbar(label="dB")
    plt.tight_layout(); plt.show()

---
### 🕐 Session 1 of 3 — *Reading Spectrograms* (~35 min)
**Goal:** learn to sight-read time–frequency pictures: tones, chirps, harmonics, percussion.
**Builds on:** [DSP Foundations](./Foundations_of_Signal_Processing_1.ipynb) S6. &nbsp; **Feeds into:** Session 2 (speech).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Reading Spectrograms</b></summary>

**Timing (~35 min).** 8 min the four glyphs · 10 min the demo, read aloud as a class · 10 min harmonic spacing and pitch · 7 min the uncertainty trade, live.

**Play the audio.** This is the one workshop in the DSP track where every cell produces something audible, and that is its whole advantage — use it. `IPython.display.Audio(melody, rate=fs)` after each synthesis cell. A student who *hears* the click and then sees the vertical stripe has learned the glyph permanently; one who only sees the plot has memorised a picture.

**Teach the four glyphs as a vocabulary, then read the demo as a class.** Horizontal line = steady tone. Stacked lines = harmonics of one note. Vertical stripe = click or percussion. Diagonal = chirp. Project the spectrogram and have the room call out what they see *before* you say anything — three harmonic stacks and two vertical stripes. Sight-reading is a skill built by doing it, not by being shown.

**The harmonic-spacing point is the one to insist on.** Pitch is not the lowest line; pitch is the *spacing* between the lines. This matters because of the missing-fundamental phenomenon: filter out the 220 Hz component entirely and the note still sounds like A3, because the ear infers pitch from the spacing of the harmonics that remain. That is why a small phone speaker with no bass response can still convey a bass line. Ask the room to predict what happens before you demonstrate it — it takes one line to notch out the fundamental and it is the most memorable minute of the session.

**Make the uncertainty principle physical.** The clicks are 80 samples — 5 ms — and they smear across the entire frequency axis, while the notes last 0.7 s and appear as thin lines. That contrast *is* the [uncertainty principle](./Foundations_of_Signal_Processing_1.ipynb), visible rather than derived. Then change `nperseg` in `show_spec` from 512 to 128 and re-run: the clicks sharpen into crisp vertical lines and the harmonic stacks blur into a smear. Same signal, same code, opposite picture. Nothing else in the curriculum makes the trade-off this tangible in fifteen seconds.

**Ask the room.** "Which window would a music transcription system use, and which would a drum-onset detector use?" Long window for pitch, short for timing. The right answer depends entirely on what you are looking for, which is why real systems compute several resolutions at once — and it previews [Time-Frequency II](./Time_Frequency_2.ipynb) if that workshop is on the syllabus.

**Note the deliberate construction.** The notes are A3 (220 Hz), C#4 (277), E4 (330) — an A-major triad, with four harmonics each and a Hann envelope so the onsets are not themselves clicks. If a student asks why the notes do not produce vertical stripes at their starts, the envelope is the answer.
</details>

## 2. The Spectrogram as Sheet Music

💡 **Intuition.** A spectrogram is *sheet music extracted from sound*: time runs right, pitch runs up, ink is energy. Horizontal lines = steady tones; stacked lines = harmonics of one note (their spacing IS the pitch); vertical stripes = clicks/percussion (uncertainty principle: sharp in time ⇒ smeared in frequency); sweeps = chirps. Learn these four glyphs and you can 'read' most sounds before hearing them.

In [2]:
t = np.arange(0, 2.5, 1/fs)
melody = np.zeros_like(t)
# three notes with harmonics (a mini 'instrument')
for start, f0 in [(0.1, 220), (0.9, 277), (1.7, 330)]:
    seg = (t >= start) & (t < start + 0.7)
    for h, amp in [(1, 1.0), (2, 0.5), (3, 0.25), (4, 0.12)]:
        melody[seg] += amp * np.sin(2*np.pi*h*f0*t[seg])
    melody[seg] *= np.hanning(seg.sum())            # note envelope
# percussion: two clicks
for click_t in [0.5, 1.3]:
    idx = int(click_t * fs)
    melody[idx:idx+80] += 2.0 * rng.standard_normal(80) * np.exp(-np.arange(80)/20)

show_spec(melody, "read it: 3 harmonic notes (A3, C#4, E4) + 2 clicks")

/tmp/ipykernel_2030389/3734263403.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Read the picture before reading this. Three groups of evenly-stacked horizontal lines are the three notes, each with four harmonics; two vertical stripes at 0.5 s and 1.3 s are the clicks. Those are two of the four glyphs, and between them they cover most of what real spectrograms contain.

**Pitch is the spacing, not the lowest line.** Each note shows lines at $f_0, 2f_0, 3f_0, 4f_0$, and the *gap* between adjacent lines equals $f_0$. This is worth insisting on because of the **missing fundamental**: filter out the 220 Hz component entirely and the note still sounds like A3, since the ear infers pitch from harmonic spacing rather than from the presence of the lowest partial. It is why a small phone speaker with no real bass response still conveys a bass line, and it is the reason Session 2's pitch estimator works by looking for periodicity rather than for a peak.

The three notes here are A3 (220 Hz), C#4 (277), E4 (330) — an A-major triad. Their harmonic stacks are identical in structure and differ only in spacing.

**The clicks are the uncertainty principle, drawn.** Each is 80 samples, about 5 ms, and each smears across the *entire* frequency axis. The notes last 0.7 s and appear as thin, well-defined lines. Perfectly localised in time means maximally spread in frequency, and vice versa — the [theorem from Foundations 1](./Foundations_of_Signal_Processing_1.ipynb), now as a picture rather than an inequality.

The striking part is that this is a property of the *analysis*, not of the signal. Change `nperseg` in `show_spec` from 512 to 128 and re-run: the clicks sharpen into crisp vertical lines while the harmonic stacks blur into an indistinct smear. Nothing about the sound changed — only the window through which we looked at it. A spectrogram is not a neutral photograph of a signal; it is one point on a resolution trade-off, and choosing the window is choosing what you are able to see. Music transcription wants a long window, drum-onset detection a short one, and systems that need both compute several in parallel.

One construction detail: each note is multiplied by a Hann envelope, so its onset is gradual. Without that, the abrupt starts would themselves produce vertical stripes — a click is simply an amplitude discontinuity, and a note that switches on instantaneously contains one.

---
### 🕐 Session 2 of 3 — *Speech: the Source-Filter Model* (~40 min)
**Goal:** synthesize vowels from scratch; estimate pitch and formants from a signal.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (effects).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Speech — the Source-Filter Model</b></summary>

**Timing (~40 min).** 10 min the source-filter factorisation · 10 min synthesising vowels · 10 min pitch and formant estimation · 10 min what the numbers actually show.

**Board first — the two-stage instrument.** Vocal cords buzz at $f_0$ (the *source*); the vocal tract resonates and colours that buzz (the *filter*). Then the sentence that makes the model click: **vowels are filter settings, not pitches.** You can sing /a/ at any pitch and it stays /a/; change your tongue position at constant pitch and it becomes /i/. Have the room do it — sing a steady note and morph /a/ → /i/ → /u/ without changing pitch. Thirty seconds of everyone humming teaches the factorisation better than any diagram.

**Make the whisper argument.** Whispering has no pitch at all — the source is noise, not a pulse train — and speech remains perfectly intelligible. So the intelligible content lives in the *filter*, and pitch carries prosody and identity instead. This is why the model is the basis of LPC compression: transmit the filter coefficients accurately and the source cheaply, and you have a phone codec.

**Point at the poles.** `sig.lfilter([1], [1, -2*r*cos(theta), r**2], x)` is a resonator, and its denominator is a complex-conjugate pole pair at radius $r$, angle $\theta$. So a formant *is* a pole, its frequency is the pole angle and its bandwidth is set by how close the pole sits to the unit circle. Students who did [Filter Design](./Filter_Design.ipynb) already own this; the vocal tract is an all-pole filter and the anatomy maps onto a pole-zero plot. That connection is the reason this workshop sits where it does in the track.

**The pitch result contains a small trap worth exploiting.** It prints 120.3 Hz against a stated 120 Hz, and the natural reading is "0.3 Hz of estimation error." That reading is wrong. `src[::int(fs/f0)]` uses `int(16000/120) = 133`, so the signal actually produced has period 133 samples and pitch $16000/133 = 120.3008$ Hz. The estimator recovered the true pitch **exactly**; the discrepancy is in the synthesiser. Ask the room where the error is before telling them — it is an excellent lesson in checking what your ground truth actually is rather than what you intended it to be.

**Read the formant errors as physics, not noise.** /a/ comes back as [723, 1088, 2365] against [730, 1090, 2440], and /i/ as [257, 2288, 3010] against [270, 2290, 3010]. The middle formants are near-perfect; the errors sit at the extremes — F3 of /a/ is 3.1% low, F1 of /i/ is 4.8% low. Two reasons worth giving: broad formants (larger bandwidth means a pole further from the unit circle, hence a less sharply defined peak) and LPC order 10 giving only five pole pairs to cover the whole spectrum, so the fit economises where it can. Raising `order` to 12 or 14 visibly improves F3, and is a good live experiment.

**Ask the room.** "Why does LPC use an all-pole model rather than poles and zeros?" Because the vocal tract genuinely is close to all-pole — a tube with resonances — so the model matches the physics. Nasals are the exception (a side branch introduces zeros), which is precisely where LPC-based vocoders sound worst.
</details>

## 3. How Speech Works

💡 **Intuition.** Speech is a two-stage instrument: the **source** (vocal cords buzzing at the pitch $f_0$, or noise for whispers/fricatives) drives a **filter** (the vocal tract, whose resonances — *formants* — are shaped by your tongue and lips). Vowels are *filter settings*: /a/ vs /i/ differ in formant positions, not pitch. This source-filter factorization is the basis of vocoders, LPC compression (your phone), and autotune.

In [3]:
def vowel(f0, formants, dur=0.6, fs=fs):
    """Synthesize a vowel: glottal pulse train through formant resonators."""
    n_samp = int(dur * fs)
    src = np.zeros(n_samp)
    src[::int(fs/f0)] = 1.0                          # impulse train at the pitch
    x = src.copy()
    for fc, bw in formants:                          # cascade of resonators (poles!)
        r = np.exp(-np.pi * bw / fs)
        theta = 2*np.pi*fc/fs
        x = sig.lfilter([1], [1, -2*r*np.cos(theta), r**2], x)
    return x / np.abs(x).max()

# classic formant tables: /a/ (father) vs /i/ (see)
a_sound = vowel(120, [(730, 90), (1090, 110), (2440, 170)])
i_sound = vowel(120, [(270, 60), (2290, 100), (3010, 180)])
both = np.concatenate([a_sound, np.zeros(1600), i_sound])
show_spec(both, "synthetic /a/ then /i/: same pitch (harmonic spacing), different formants (bright bands)")

/tmp/ipykernel_2030389/3734263403.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two synthetic vowels, and the spectrogram shows exactly the factorisation the model predicts. Both halves have **identical harmonic spacing** — same pitch, 120 Hz, same source. What differs is *which* harmonics are loud: /a/ has bright bands near 730 and 1090 Hz sitting close together, while /i/ has one very low band near 270 Hz and a widely separated pair up at 2290 and 3010 Hz.

That is the whole source-filter model in one picture. The **source** sets the harmonic comb — its spacing is the pitch, and every harmonic is present. The **filter** sets the envelope — which harmonics get amplified. Vowel identity lives entirely in the envelope, so /a/ and /i/ at the same pitch differ in formants, not in their harmonic structure. Sing a steady note and morph between the two vowels: the pitch does not move, the resonances do.

The strongest evidence for this factorisation is whispering. A whisper has no pitch at all — the source is noise rather than a pulse train, so there is no comb — and speech remains perfectly intelligible. The intelligible content is in the filter; pitch carries prosody and speaker identity. This is exactly why LPC codecs work: send the filter coefficients accurately and describe the source cheaply, and you have compressed speech to a few kbit/s. Your phone call is running this model.

**And the code is doing DSP the room already knows.** Each formant is created by `sig.lfilter([1], [1, -2*r*cos(theta), r**2], x)` — a denominator with a complex-conjugate **pole pair**. The pole angle $\theta = 2\pi f_c/f_s$ sets the formant frequency, and the radius $r = e^{-\pi\,\mathrm{bw}/f_s}$ sets its bandwidth, with poles nearer the unit circle giving sharper resonances. So a formant is a pole, the vocal tract is an all-pole filter, and the [pole-zero plots from Filter Design](./Filter_Design.ipynb) are a map of anatomy. The all-pole assumption is physically motivated too — a tube with resonances has poles and few zeros — which is why LPC-based vocoders sound worst on nasals, where a side branch does introduce zeros.

The next cell inverts this: given only the waveform, recover the pitch and the formants that produced it.

In [4]:
# Pitch estimation by autocorrelation: the lag where the signal rhymes with itself
def estimate_pitch(x, fs=fs, fmin=60, fmax=400):
    x = x - x.mean()
    r = np.correlate(x, x, "full")[len(x)-1:]
    lo, hi = int(fs/fmax), int(fs/fmin)
    return fs / (lo + np.argmax(r[lo:hi]))

print(f"estimated pitch of /a/: {estimate_pitch(a_sound):.1f} Hz  (synthesized at 120 Hz)")

# Formant estimation via LPC (all-pole fit — Wiener/Yule-Walker in disguise)
def lpc_formants(x, order=10, fs=fs):
    x = x * np.hanning(len(x))
    r = np.correlate(x, x, "full")[len(x)-1:len(x)+order]
    from scipy.linalg import solve_toeplitz
    a = solve_toeplitz(r[:-1], r[1:])                # Yule-Walker
    roots = np.roots(np.concatenate([[1], -a]))
    roots = roots[(roots.imag > 0)]
    freqs = np.sort(np.angle(roots) * fs / (2*np.pi))
    return freqs[freqs > 90][:3]

print("estimated /a/ formants:", np.round(lpc_formants(a_sound)), " (target ≈ [730, 1090, 2440])")
print("estimated /i/ formants:", np.round(lpc_formants(i_sound)), " (target ≈ [270, 2290, 3010])")

estimated pitch of /a/: 120.3 Hz  (synthesized at 120 Hz)
estimated /a/ formants: [ 723. 1088. 2365.]  (target ≈ [730, 1090, 2440])
estimated /i/ formants: [ 257. 2288. 3010.]  (target ≈ [270, 2290, 3010])


**What just happened.** Both halves of the source-filter model, recovered from the waveform alone.

**The pitch estimate is exact — and the printed numbers hide that.** It reports 120.3 Hz against a synthesis target of 120 Hz, which looks like 0.3 Hz of error. It is not. Look at `src[::int(fs/f0)]`: with $f_s = 16000$ and $f_0 = 120$, `int(16000/120)` truncates to **133** samples, so the signal we actually built has a period of 133 samples and a true pitch of $16000/133 = 120.3008$ Hz. The autocorrelation estimator found precisely that. The 0.3 Hz lives in the *synthesiser's* integer rounding, not in the estimate.

Worth pausing on, because the mistake is general: the ground truth is what the code produced, not what the variable was named. An error attributed to the wrong stage sends you tuning an estimator that was already perfect. Always check what your reference actually is.

The method itself is the missing-fundamental idea from Session 1 made into an algorithm. Autocorrelation asks "at what lag does this signal rhyme with itself?" — it looks for *periodicity*, not for a spectral peak, so it still works when the fundamental is weak or absent entirely.

**The formants come back close, and the errors are informative.** For /a/: [723, 1088, 2365] against [730, 1090, 2440]. For /i/: [257, 2288, 3010] against [270, 2290, 3010]. The middle formants are nearly exact (F2 of /a/ is 0.2% off, F2 of /i/ 0.1%), while the errors concentrate at the extremes — F3 of /a/ is 3.1% low and F1 of /i/ is 4.8% low.

Two mechanisms, both worth knowing. Broad formants are estimated less precisely: bandwidth 170 Hz puts the pole further from the unit circle, so the spectral peak is flatter and its location less sharply determined than a 60 Hz formant's. And LPC order 10 provides only five pole pairs to cover the entire 0–8 kHz range, so the fit spends its poles where the energy is and economises at the top, pulling F3 downward. Raise `order` to 14 and F3 improves visibly — a one-character experiment worth running.

**What `lpc_formants` is really doing.** `solve_toeplitz(r[:-1], r[1:])` is the Yule–Walker equations — the same normal equations as the Wiener filter in [Statistical SP](./Statistical_Signal_Processing.ipynb) and the same AR fit as [Classical Forecasting](../Intro_Time_Series/Classical_Forecasting.ipynb). Then `np.roots` finds the poles and their angles become frequencies. Speech coding, spectral estimation, and time-series forecasting are running identical mathematics; only the interpretation of the poles differs.

Note finally that estimation here is on *synthetic* speech that exactly obeys the all-pole model. Real speech has a glottal source with its own spectral tilt, nasal zeros, and noise — so real formant tracking is substantially harder, and the clean agreement above reflects a matched model as much as a good estimator.

---
### 🕐 Session 3 of 3 — *Effects Are Filters* (~35 min)
**Goal:** build reverb, robot voice, and a pitch shifter — and see each as a DSP primitive.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Effects Are Filters</b></summary>

**Timing (~35 min).** 5 min the table · 8 min reverb · 8 min ring modulation · 10 min the pitch shifter · 4 min buffer.

**Lead with the table and the deflation.** The entire studio effects industry is convolution, modulation, filtering, and resampling — with good marketing. Students find this genuinely surprising and a little disappointing, which is the right reaction: it means everything they have already learned applies directly to something they care about. Work the table before any code.

**Reverb is just convolution — make the impulse response tangible.** The IR is "what the room does to a clap." Convolving a dry signal with it places that signal in the room. Then note that convolution reverb in real plugins uses *measured* IRs — people record claps or sine sweeps in concert halls and cathedrals and sell the results. Ask why `ir[0] = 1.0` is there: it preserves the direct (unreverberated) path, and without it you would hear only reflections, as if the source were behind you.

**Ring modulation is the trig identity, and the room can derive it.** $\sin A \sin B = \tfrac12[\cos(A-B) - \cos(A+B)]$ — every frequency component splits into a sum and difference pair around the carrier. Have them predict what the spectrogram will look like before running: each harmonic becomes two, displaced by ±70 Hz, and crucially the *spacing* is no longer harmonic, which is why it sounds metallic and inhuman rather than merely pitch-shifted. This is also the [Digital Communications](./Digital_Communications.ipynb) mixer, so the same operation is a Dalek voice and a superheterodyne receiver.

**The pitch shifter is the one worth slowing down for.** Two stages: time-stretch without changing pitch (analyse with one hop, synthesise with a *smaller* hop), then resample to restore the original duration, which raises the pitch. Ask why you cannot simply resample — because that changes speed and pitch together, the tape-machine effect, and shifts the formants too, which is what produces the chipmunk sound. Keeping formants fixed while moving pitch is exactly what Session 2's factorisation says you should want, and it is why professional pitch correction operates on the source and filter separately.

**Be honest that this shifter is crude.** Using `istft` with a mismatched `noverlap` stretches the signal but does not correct the phase between frames, so the result has the characteristic "phasiness" of a naive phase vocoder. Real implementations propagate phase across frames or use transient-preserving variants. It demonstrates the mechanism; it is not a good pitch shifter, and students should not benchmark against it.

**Play everything.** Reverb, robot, and pitch shift are effects students recognise from music. Hearing them emerge from four lines of DSP is the payoff of the whole workshop — `IPython.display.Audio(y, rate=fs)` on each. The export line in the cell is there so they can take the results away.

**Close with the homework.** Record a real voice on a phone, load it, and re-run every cell in the workshop. Everything here works on real audio, and the gap between synthetic and real is where the interesting difficulties live.
</details>

## 4. The Effects Rack

Every studio effect is a signal-processing primitive wearing a costume:

| Effect | DSP primitive |
|---|---|
| Echo/reverb | convolution with a (sparse/dense) impulse response |
| Robot voice | ring modulation (multiply by a carrier) |
| Wah / EQ | time-varying / fixed [filters](./Filter_Design.ipynb) |
| Pitch shift | STFT: stretch time, then resample ([multirate](./Foundations_of_Signal_Processing_2.ipynb)) |

In [5]:
voice = both                                        # our /a/-/i/ 'phrase'

# 1) Reverb: convolve with an exponentially decaying random impulse response
ir = rng.standard_normal(int(0.4*fs)) * np.exp(-np.arange(int(0.4*fs)) / (0.12*fs))
ir[0] = 1.0
reverbed = sig.fftconvolve(voice, 0.4*ir)[:len(voice)]

# 2) Robot: ring-modulate with a 70 Hz carrier
robot = voice * np.sin(2*np.pi*70*np.arange(len(voice))/fs)

# 3) Pitch shift up a fourth: phase-vocoder-lite = time-stretch (STFT hop trick) + resample
f_, t_, S = sig.stft(voice, fs=fs, nperseg=1024, noverlap=768)
_, stretched = sig.istft(S, fs=fs, nperseg=1024, noverlap=896)     # smaller synthesis hop → slower
shifted = sig.resample(stretched, len(voice))                       # resample back → higher pitch

fig, axes = plt.subplots(1, 3, figsize=(10.5, 2.5))
for ax, (y, name) in zip(axes, [(reverbed, "reverb"), (robot, "robot (sidebands!)"), (shifted, "pitch-shifted")]):
    fq, tq, Sq = sig.stft(y, fs=fs, nperseg=512)
    ax.pcolormesh(tq, fq, 20*np.log10(np.abs(Sq)+1e-8), shading="auto", vmin=-100, vmax=-20)
    ax.set_title(name); ax.set_ylim(0, 4000)
plt.tight_layout(); plt.show()
print("export any of these:  from scipy.io import wavfile;  wavfile.write('out.wav', fs, (y*32767).astype('int16'))")

export any of these:  from scipy.io import wavfile;  wavfile.write('out.wav', fs, (y*32767).astype('int16'))


/tmp/ipykernel_2030389/2223069117.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three recognisable studio effects, each about four lines, and each a DSP primitive the room already owns.

**Reverb (left) is convolution, nothing more.** The impulse response is "what the room does to a clap" — here an exponentially decaying noise burst — and convolving the dry voice with it places that voice in the room. The spectrogram shows every event smeared *rightward* in time, energy trailing after each onset, with the harmonic structure preserved because convolution with a broadband IR does not move frequencies. Note `ir[0] = 1.0`: that spike preserves the direct path, and without it you would hear only reflections, as though the speaker were facing away. Commercial convolution reverbs work exactly this way, using impulse responses *measured* in real concert halls and cathedrals.

**Ring modulation (middle) is a trigonometric identity.** $\sin A\sin B = \tfrac12[\cos(A-B) - \cos(A+B)]$, so multiplying by a 70 Hz carrier splits every component into a sum-and-difference pair. The spectrogram shows the sidebands clearly — each original harmonic has become two, displaced by ±70 Hz.

The reason it sounds *inhuman* rather than merely pitch-shifted is the point worth extracting. The original partials sat at $f_0, 2f_0, 3f_0,\dots$ — a harmonic series. After modulation they sit at $nf_0 \pm 70$, and those are **no longer integer multiples of anything**. The ear identifies pitch from harmonic spacing (Session 1), so a spectrum with no consistent spacing has no coherent pitch, and we hear a metallic, robotic timbre. Incidentally, this same operation is the mixer in a superheterodyne receiver from [Digital Communications](./Digital_Communications.ipynb) — a Dalek voice and a radio front end are one circuit.

**Pitch shifting (right) is multirate processing in two moves.** First time-stretch without altering pitch, by analysing at one hop and re-synthesising at a smaller one; then resample back to the original duration, which raises the pitch. Ask why you cannot just resample directly: that changes speed and pitch together — the tape-machine effect — and it also drags the *formants* along, which is what makes the chipmunk sound. Session 2 explains why that is wrong: formants define vowel identity, so shifting them changes *who is speaking*, not just the note. Real pitch correction moves the source and leaves the filter alone.

**Be honest about this implementation.** Reusing `istft` with a mismatched `noverlap` stretches the signal but does not reconcile phase between frames, so the output has the characteristic "phasiness" of a naive phase vocoder. Proper implementations propagate phase across frames and handle transients separately. This demonstrates the mechanism; it is not a good pitch shifter.

**And the deflationary conclusion.** Reverb is convolution, robot voice is modulation, pitch shift is resampling, EQ and wah are filters. The whole effects industry is the DSP in this track, packaged and marketed. That should be encouraging rather than disappointing: you already have the tools, and the export line in this cell means you can take the results away and listen to them.

## 5. Conclusion

Spectrograms are readable sheet music; speech factors into source × filter (pitch × formants); and the entire effects industry is convolution, modulation, filtering, and resampling with good marketing. Record a real voice and rerun every cell — that's the homework.

---
## Where next

- [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) — classifying exactly these spectrograms.
- [Foundations 2](./Foundations_of_Signal_Processing_2.ipynb) — the multirate machinery under the pitch shifter.
- [Array Processing](./Array_Processing.ipynb) — what the *second* microphone buys you.